# Beispiele: Clustering und Anomalien

k-Means mit Elbow und Silhouette, hierarchisches Clustering, DBSCAN, GMM, Cluster-Profile und Isolation Forest. Beim Clustering gibt es kein `y`. Die Beispiele erzeugen meist Daten mit bekannten Gruppen, damit man sieht, ob ein Verfahren sie findet.

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten sind in scikit-learn eingebaut oder werden erzeugt.
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/04_Clustering_Anomalien.ipynb) (ein Google-Konto genügt, solange das Repo öffentlich ist; ist es privat, in Colab mit GitHub anmelden und «Include private repos» anhaken).
- **Navigation:** Das Inhaltsverzeichnis links in JupyterLab springt zu jedem Abschnitt. Zurück zur Übersicht: Lernapp, Kapitel «Übersicht».

In [ ]:
# Colab has scikit-learn, pandas and matplotlib; umap and imblearn are installed there on demand
import importlib.util, subprocess, sys
fehlend = [pip for mod, pip in [("umap", "umap-learn"), ("imblearn", "imbalanced-learn")] if importlib.util.find_spec(mod) is None]
if fehlend and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fehlend], check=True)
elif fehlend:
    print("Für einzelne Beispiele fehlt: pip install " + " ".join(fehlend))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (load_diabetes, load_breast_cancer, load_wine, load_digits,
                              make_moons, make_blobs, make_classification)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, silhouette_score,
                             adjusted_rand_score)

SEED = 42
plt.rcParams["figure.figsize"] = (6, 3.5)
pd.set_option("display.precision", 3)

## k-Means und die Wahl von k

Vier Gruppen. Elbow (Inertia) und Silhouette für k = 2 bis 8.

In [ ]:
from sklearn.cluster import KMeans

Xb, yb = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=SEED)
Xb = StandardScaler().fit_transform(Xb)

ks, inertia, sil = range(2, 9), [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xb)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xb, km.labels_))
print("Silhouette:", {k: round(s, 3) for k, s in zip(ks, sil)})

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3))
a1.plot(ks, inertia, "o-"); a1.set_title("Elbow: Inertia"); a1.set_xlabel("k")
a2.plot(ks, sil, "o-"); a2.set_title("Silhouette (höher = besser)"); a2.set_xlabel("k")
km4 = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(Xb)
a3.scatter(Xb[:, 0], Xb[:, 1], c=km4.labels_, s=8, cmap="tab10")
a3.scatter(*km4.cluster_centers_.T, c="black", marker="x", s=80); a3.set_title("k = 4")
plt.tight_layout(); plt.show()

**Deutung:** Der Knick der Inertia und das Maximum der Silhouette liegen beide bei k = 4. Bei echten Daten sind die beiden oft weniger eindeutig. Dann entscheidet, welche Gruppierung inhaltlich Sinn ergibt.

## Hierarchisches Clustering

Das Dendrogramm zeigt alle Zusammenschlüsse. Ein grosser senkrechter Abstand vor dem nächsten Zusammenschluss ist ein guter Schnitt.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

probe = Xb[::10]   # 50 points, so the dendrogram stays readable
dendrogram(linkage(probe, method="ward"), no_labels=True)
plt.title("Dendrogramm (Ward)"); plt.ylabel("Abstand"); plt.show()

agg = AgglomerativeClustering(n_clusters=4, linkage="ward").fit(Xb)
print("Übereinstimmung mit k-Means (ARI, 1 = identisch):", round(adjusted_rand_score(km4.labels_, agg.labels_), 3))

## DBSCAN gegen k-Means bei krummen Formen

Halbmonde sind nicht rund. k-Means schneidet sie gerade durch, DBSCAN folgt der Dichte und markiert Ausreisser mit `-1`.

In [ ]:
from sklearn.cluster import DBSCAN

Xd, yd = make_moons(n_samples=400, noise=0.07, random_state=SEED)
Xd = StandardScaler().fit_transform(Xd)
km_d = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(Xd)
db_d = DBSCAN(eps=0.3, min_samples=5).fit_predict(Xd)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for ax, lab, titel in [(axes[0], km_d, "k-Means"), (axes[1], db_d, "DBSCAN")]:
    ax.scatter(Xd[:, 0], Xd[:, 1], c=lab, s=8, cmap="tab10")
    ax.set_title(f"{titel}: ARI {adjusted_rand_score(yd, lab):.2f}")
plt.show()
print("DBSCAN: Cluster", len(set(db_d) - {-1}), "· Rauschpunkte", int(np.sum(db_d == -1)))

In [ ]:
# eps ist der heikle Regler: zu klein = alles Rauschen, zu gross = ein einziger Cluster
for eps in [0.1, 0.2, 0.3, 0.5, 1.0]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xd)
    print(f"eps={eps}: {len(set(lab) - {-1})} Cluster, {int(np.sum(lab == -1))} Rauschpunkte")

## GMM (Gaussian Mixture)

Länglich gezogene Gruppen. GMM passt Ellipsen an und liefert pro Punkt Wahrscheinlichkeiten, k-Means nur harte Zuordnungen mit runden Grenzen.

In [ ]:
from sklearn.mixture import GaussianMixture

Xg, yg = make_blobs(n_samples=600, centers=3, random_state=170)
Xg = Xg @ np.array([[0.6, -0.6], [-0.4, 0.8]])   # stretch the blobs into ellipses

km_g = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(Xg)
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=SEED).fit(Xg)
gm_g = gmm.predict(Xg)
print(f"ARI k-Means: {adjusted_rand_score(yg, km_g):.2f} · ARI GMM: {adjusted_rand_score(yg, gm_g):.2f}")

unsicher = gmm.predict_proba(Xg).max(axis=1) < 0.8
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].scatter(Xg[:, 0], Xg[:, 1], c=km_g, s=8, cmap="tab10"); axes[0].set_title("k-Means")
axes[1].scatter(Xg[:, 0], Xg[:, 1], c=gm_g, s=8, cmap="tab10")
axes[1].scatter(Xg[unsicher, 0], Xg[unsicher, 1], facecolors="none", edgecolors="black", s=30)
axes[1].set_title("GMM (Kreis = unsicher)"); plt.show()

## Cluster prüfen und erklären

Ein Cluster ist erst nützlich, wenn man ihn beschreiben kann. Profil: Mittelwerte der Originalmerkmale pro Cluster. Hier mit den Weinen, deren echte Sorte nur zur Kontrolle dient.

In [ ]:
Xw, yw = load_wine(return_X_y=True, as_frame=True)
Xw_z = StandardScaler().fit_transform(Xw)
cl = KMeans(n_clusters=3, n_init=10, random_state=SEED).fit_predict(Xw_z)
profil = Xw.assign(cluster=cl).groupby("cluster")[["alcohol", "malic_acid", "flavanoids", "color_intensity", "proline"]].mean()
print(profil.round(2))
print()
print("Kontrolle gegen die echte Sorte:")
print(pd.crosstab(cl, yw, rownames=["Cluster"], colnames=["Sorte"]))

## Anomalien: Isolation Forest

Simulierte Zahlungen (Betrag, Uhrzeit). 15 auffällige Zahlungen werden eingestreut: hohe Beträge mitten in der Nacht. Wie viele findet der Isolation Forest?

In [ ]:
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(SEED)
normal = pd.DataFrame({"betrag": rng.lognormal(4, 0.5, 1000), "stunde": rng.normal(14, 3, 1000).clip(0, 23)})
auffaellig = pd.DataFrame({"betrag": rng.uniform(800, 2000, 15), "stunde": rng.uniform(1, 4, 15)})
zahlungen = pd.concat([normal, auffaellig], ignore_index=True)
echt = np.r_[np.zeros(1000), np.ones(15)].astype(bool)

iso = IsolationForest(contamination=0.015, random_state=SEED).fit(zahlungen)
markiert = iso.predict(zahlungen) == -1
print("markiert:", markiert.sum(), "· davon wirklich auffällig:", (markiert & echt).sum(), "von", echt.sum())

plt.scatter(zahlungen.betrag, zahlungen.stunde, c=markiert, s=8, cmap="coolwarm")
plt.xscale("log"); plt.xlabel("Betrag (log)"); plt.ylabel("Stunde"); plt.title("Rot = vom Isolation Forest markiert"); plt.show()

**Deutung:** `contamination` legt fest, wie viele Punkte markiert werden. Ist der Wert zu gross, gibt es falschen Alarm. Ist er zu klein, werden Fälle verpasst. Der Score ist ein Hinweis zum Nachprüfen, kein Beweis.